# Notebook 04 — Inference and Automatic Metrics

Goal: load both adapters (Track A and Track B), generate predictions on the
same held-out test rows, extract final answers, and compute EM / ROUGE-L
plus optional BERTScore / sacreBLEU.

## Required Kaggle environment
- Accelerator: **GPU T4 x1**
- Internet: **On**
- Kaggle Secrets: `HF_TOKEN` (read scope for private adapter repos), `WANDB_API_KEY`

## 1. Bootstrap (clone, install, secrets)


In [ ]:
import warnings
warnings.filterwarnings('ignore', message='.*AttentionMaskConverter.*')
warnings.filterwarnings('ignore', category=FutureWarning, module='transformers')

import os, sys, subprocess, json
import pandas as pd
from pathlib import Path

REPO_URL = 'https://github.com/abhishek1998s/medical-reasoning-llm.git'
REPO_DIR = '/kaggle/working/medical-reasoning-llm'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
!rm -rf /kaggle/working/unsloth_compiled_cache
!pip install -q --upgrade \
    unsloth==2026.4.8 \
    transformers==5.5.0 \
    trl==0.24.0 \
    peft==0.19.1 \
    bitsandbytes==0.49.2 \
    accelerate==1.13.0 \
    datasets==4.3.0 \
    wandb==0.19.4 \
    pyyaml

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login as hf_login

secrets = UserSecretsClient()

def _try_get(name):
    try:
        return secrets.get_secret(name)
    except Exception as e:
        print(f'  [skip] {name}: {e.__class__.__name__}')
        return None

os.environ['HF_TOKEN']      = _try_get('HF_TOKEN')      or ''
os.environ['WANDB_API_KEY'] = _try_get('WANDB_API_KEY') or ''

print('HF_TOKEN set:     ', bool(os.environ['HF_TOKEN']))
print('WANDB_API_KEY set:', bool(os.environ['WANDB_API_KEY']))

if os.environ['HF_TOKEN']:
    hf_login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
    print('HF login OK')

## 2. Load Shared Test Split


In [ ]:
import yaml
from datasets import load_dataset
from transformers import AutoTokenizer
from src.splits import shuffle_filter_split

cfg = yaml.safe_load(open('configs/experiment_config.yaml', encoding='utf-8'))
tok = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct')
ds = load_dataset(cfg['dataset']['name'], split=cfg['dataset']['split'],
                  token=os.environ['HF_TOKEN'] or None)
_, _, test_ds = shuffle_filter_split(
    ds,
    shuffle_seed=cfg['dataset']['shuffle_seed'],
    num_train=cfg['dataset']['num_train'],
    num_val=cfg['dataset']['num_val'],
    num_test=cfg['dataset']['num_test'],
    tokenizer=tok,
    max_total_tokens=3500,
    max_rows=cfg['dataset'].get('max_rows'),
)
print('test rows:', len(test_ds))

# Save frozen split indices so NB05/06 can reload without config dependency
import json as _json
_split_path = Path('outputs/test_split_indices.json')
_split_path.parent.mkdir(parents=True, exist_ok=True)
_split_path.write_text(_json.dumps({
    'indices': (
        test_ds._indices.to_pylist()
        if hasattr(test_ds, '_indices') and test_ds._indices is not None
        else list(range(len(test_ds)))
    ),
    'shuffle_seed': cfg['dataset']['shuffle_seed'],
    'num_test': cfg['dataset']['num_test'],
    'max_rows': cfg['dataset'].get('max_rows'),
}, indent=2))
print(f'Split indices saved -> {_split_path}')

## 3. Generate Predictions


In [ ]:
import gc
import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from peft import PeftModel
from src.inference import build_prediction_row, generate_with_logging

hf_token = os.environ.get('HF_TOKEN') or None
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

def load_model(adapter_id):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=cfg['model']['name'],
        max_seq_length=cfg['model']['max_seq_length'],
        dtype=None,
        load_in_4bit=True,
    )
    tokenizer = get_chat_template(tokenizer, chat_template=cfg['model']['chat_template'])
    # Pass token explicitly so private Hub repos are accessible.
    model = PeftModel.from_pretrained(model, adapter_id, token=hf_token)
    FastLanguageModel.for_inference(model)
    return model, tokenizer

def run_track(track_name, adapter_id, out_csv):
    print(f'\n[{track_name}] loading adapter: {adapter_id}')
    model, tokenizer = load_model(adapter_id)
    rows = []
    cfg_key = 'trackA' if track_name == 'A' else 'trackB'
    max_new = cfg['inference']['max_new_tokens'][cfg_key]
    for i, row in enumerate(test_ds):
        user = next(m for m in row['messages'] if m['role'] == 'user')
        asst = next(m for m in row['messages'] if m['role'] == 'assistant')
        gen = generate_with_logging(
            model, tokenizer, user['content'],
            max_new_tokens=max_new,
            temperature=cfg['inference']['temperature'],
            do_sample=cfg['inference']['do_sample'],
            repetition_penalty=cfg['inference']['repetition_penalty'],
            device=device,
        )
        rows.append(build_prediction_row(
            sample_id=i,
            question=user['content'],
            reference=(asst.get('content') or '').strip(),
            track_name=track_name,
            model_id=cfg['model']['name'],
            adapter_id=adapter_id,
            generation=gen,
        ))
        if (i + 1) % 10 == 0:
            print(f'  {i + 1}/{len(test_ds)} done')
    Path(out_csv).parent.mkdir(parents=True, exist_ok=True)
    result = pd.DataFrame(rows)
    result.to_csv(out_csv, index=False)
    print(f'  saved -> {out_csv}')
    # Free GPU memory before loading the next adapter
    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return result

adapter_a = f"{cfg['hub']['username']}/{cfg['hub']['repos']['trackA']}"
adapter_b = f"{cfg['hub']['username']}/{cfg['hub']['repos']['trackB']}"

track_a = run_track('A', adapter_a, 'outputs/trackA/predictions.csv')
track_b = run_track('B', adapter_b, 'outputs/trackB/predictions.csv')

## 4. Automatic Metrics


In [ ]:
from src.data_formatting import extract_answer_for_scoring
from src.metrics import compute_core_metrics

def score_file(path, track):
    df = pd.read_csv(path)
    preds = [extract_answer_for_scoring(p, track) for p in df['prediction']]
    refs  = list(df['reference'])
    core  = compute_core_metrics(preds, refs)
    core['mean_output_tokens']    = round(float(df['output_tokens'].mean()), 2)
    core['mean_generation_time_s']= round(float(df['generation_time_s'].mean()), 3)
    core['mean_tokens_per_sec']   = round(float(df['tokens_per_sec'].mean()), 2)
    return core

summary = {
    'trackA': score_file('outputs/trackA/predictions.csv', 'A'),
    'trackB': score_file('outputs/trackB/predictions.csv', 'B'),
}
Path('outputs/metrics_summary.json').write_text(
    json.dumps(summary, indent=2), encoding='utf-8'
)

import pandas as pd
pd.DataFrame(summary).T

In [ ]:
# Richer inference diagnostics: truncation, finish-reason, length percentiles
from src.metrics import compute_operational_stats

for track_name, csv_path in [('A', 'outputs/trackA/predictions.csv'),
                               ('B', 'outputs/trackB/predictions.csv')]:
    df = pd.read_csv(csv_path)
    stats = compute_operational_stats(df)
    print(f'\n=== Track {track_name} Inference Diagnostics ===')
    print(f"  truncation_rate:      {stats.get('truncation_rate', 'N/A'):.1%}")
    print(f"  empty_prediction_rate:{stats.get('empty_prediction_rate', 'N/A'):.1%}")
    print(f"  finish_reason_dist:   {stats.get('finish_reason_dist', {})}")
    print(f"  output_tokens p50/p90/p99: "
          f"{stats.get('output_tokens_p50','?')} / "
          f"{stats.get('output_tokens_p90','?')} / "
          f"{stats.get('output_tokens_p99','?')}")
    print(f"  gen_time p50/p90:     "
          f"{stats.get('generation_time_p50','?'):.3f}s / "
          f"{stats.get('generation_time_p90','?'):.3f}s")